# 05 · Matrix factorisation: ALS and BPR

**Learning objectives**

1. Explain **latent factor models**: users and items as vectors in a shared "taste space".
2. Derive the **implicit ALS** update, and understand *confidence* vs *preference*.
3. Understand **BPR**, which optimises the ranking directly.
4. Tune factors, regularisation and α on validation, and inspect what the embeddings learned.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from recsys.data import load_or_synthesize
from recsys.pipeline import prepare_data
from recsys.metrics import evaluate, evaluate_many
from recsys.models import ALSRecommender, BPRRecommender, ItemKNNRecommender, PopularityRecommender
from recsys.plotting import set_style, PALETTE, MUTED, model_color, bar_leaderboard

set_style()
events, item_categories, SOURCE = load_or_synthesize()
data = prepare_data(events)
val = prepare_data(events[events["timestamp"] < data.cutoff])  # for tuning: see notebook 04

## 1. Intuition

Item-kNN compares items **directly**. Matrix factorisation (MF) instead **compresses** the user × item matrix into two thin matrices:

$$ R \;\approx\; X\,Y^\top, \qquad X \in \mathbb{R}^{n_{users}\times f},\; Y \in \mathbb{R}^{n_{items}\times f},\; f \ll n_{items}$$

Each user and item becomes an $f$-dimensional vector, and the predicted affinity is their **dot product**. Because $f$ is small, the model has to find *shared structure* ("outdoor gear", "budget electronics"). This lets it generalise to item pairs that never co-occurred. Item-kNN cannot do that.

## 2. Implicit ALS

With implicit data the matrix has no negatives, only "observed" and "unknown". Hu, Koren & Volinsky (2008) split each cell into:

* **preference** $p_{ui} = \mathbb{1}[r_{ui} > 0]$: is there evidence of interest at all?
* **confidence** $c_{ui} = 1 + \alpha\, r_{ui}$: how much do we trust it? Unobserved cells get confidence 1. They count as weak evidence of "no".

Then minimise over **all** cells (including the unobserved ones):

$$ \min_{X,Y} \sum_{u,i} c_{ui}\,(p_{ui} - x_u^\top y_i)^2 + \lambda\Big(\sum_u\|x_u\|^2 + \sum_i\|y_i\|^2\Big) $$

**Why "alternating"?** If $Y$ is fixed, the objective is a weighted ridge regression for each user, with a closed-form solution:

$$ x_u = \big(Y^\top C^u Y + \lambda I\big)^{-1} Y^\top C^u p(u) $$

So we alternate: solve all users with items fixed, then all items with users fixed. Each step can only lower the loss.

**The speed trick.** $Y^\top C^u Y = Y^\top Y + Y^\top (C^u - I) Y$. The first term is shared by every user. The second only involves the handful of items the user touched. Open `src/recsys/models/als.py` to see it in ~15 lines.

In [ ]:
als = ALSRecommender(factors=32, regularization=1.0, alpha=10, iterations=10, verbose=True).fit(val.X_train)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(range(1, len(als.loss_history_) + 1), als.loss_history_, marker="o", color=model_color("als"))
ax.set_xlabel("iteration"); ax.set_ylabel("objective"); ax.set_title("ALS loss decreases monotonically")
plt.show()

## 3. Tuning ALS on validation

| hyperparameter | intuition |
|---|---|
| `factors` | capacity. Too few underfits; too many memorises noise (and costs $O(f^3)$ per solve) |
| `regularization` λ | shrinks the vectors. Fights overfitting, especially for users with few interactions |
| `alpha` | how much more an observed interaction counts than an unobserved one |

We use a small grid to keep the run time sane. In practice you would use random search or Bayesian optimisation.

In [ ]:
grid = []
for factors in [16, 32, 64]:
    for reg in [0.1, 1.0, 10.0]:
        for alpha in [5, 20]:
            m = ALSRecommender(factors=factors, regularization=reg, alpha=alpha, iterations=8).fit(val.X_train)
            grid.append({"factors": factors, "regularization": reg, "alpha": alpha,
                         "ndcg@10": evaluate(m, val.X_train, val.ground_truth)["ndcg@10"]})
grid = pd.DataFrame(grid).sort_values("ndcg@10", ascending=False)
best_als = grid.iloc[0][["factors", "regularization", "alpha"]].to_dict()
print("best ALS on validation:", best_als)
grid.head(8).round(4)

In [ ]:
pivot = grid.pivot_table(index="factors", columns="regularization", values="ndcg@10", aggfunc="max")
fig, ax = plt.subplots(figsize=(5, 3))
im = ax.imshow(pivot.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(pivot.columns)), [str(c) for c in pivot.columns]); ax.set_xlabel("regularization λ")
ax.set_yticks(range(len(pivot.index)), [str(i) for i in pivot.index]); ax.set_ylabel("factors")
for (r, c), v in np.ndenumerate(pivot.values):
    ax.text(c, r, f"{v:.3f}", ha="center", va="center", fontsize=9,
            color="white" if v > np.nanmean(pivot.values) else "black")
ax.grid(False); ax.set_title("Validation NDCG@10 (best α)")
plt.colorbar(im, ax=ax)
plt.show()

## 4. BPR: optimise the ranking directly

ALS regresses towards 0/1 targets. But we never use the predicted *value*, only the *order*. **Bayesian Personalised Ranking** (Rendle et al., 2009) optimises the order directly. For a user $u$, a positive item $i$ and a sampled negative $j$:

$$ \max_\Theta \sum_{(u,i,j)} \ln \sigma\big(\hat x_{ui} - \hat x_{uj}\big) - \lambda\|\Theta\|^2 $$

This is a smooth version of AUC. It is trained with SGD by sampling triples. The gradient factor is $\sigma(-(\hat x_{ui}-\hat x_{uj}))$, so triples that are already ordered correctly barely update the model.

The training AUC below is the share of sampled triples ordered correctly. It should climb quickly towards 1.

In [ ]:
bpr = BPRRecommender(factors=32, learning_rate=0.1, regularization=0.01, epochs=30, verbose=False).fit(val.X_train)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(range(1, len(bpr.auc_history_) + 1), bpr.auc_history_, color=model_color("bpr"))
ax.set_xlabel("epoch"); ax.set_ylabel("sampled training AUC"); ax.set_title("BPR learns to order positives above negatives")
plt.show()

In [ ]:
bpr_grid = []
for lr in [0.03, 0.1]:
    for reg in [0.001, 0.01, 0.05]:
        m = BPRRecommender(factors=32, learning_rate=lr, regularization=reg, epochs=30).fit(val.X_train)
        bpr_grid.append({"learning_rate": lr, "regularization": reg,
                         "ndcg@10": evaluate(m, val.X_train, val.ground_truth)["ndcg@10"]})
bpr_grid = pd.DataFrame(bpr_grid).sort_values("ndcg@10", ascending=False)
best_bpr = bpr_grid.iloc[0][["learning_rate", "regularization"]].to_dict()
print("best BPR on validation:", best_bpr)
bpr_grid.round(4)

## 5. Final comparison on test

In [ ]:
als = ALSRecommender(factors=int(best_als["factors"]), regularization=best_als["regularization"],
                     alpha=best_als["alpha"]).fit(data.X_train)
bpr = BPRRecommender(learning_rate=best_bpr["learning_rate"], regularization=best_bpr["regularization"]).fit(data.X_train)
board = evaluate_many({
    "popularity": PopularityRecommender().fit(data.X_train),
    "item_knn": ItemKNNRecommender().fit(data.X_train),
    "als": als,
    "bpr": bpr,
}, data.X_train, data.ground_truth)
display(board.round(4))
bar_leaderboard(board, "ndcg@10", title="Test NDCG@10")
plt.show()

Don't be surprised if item-kNN is competitive with, or beats, MF on RetailRocket. Short, intent-driven e-commerce sessions favour direct co-occurrence. MF tends to shine with richer per-user histories and for **coverage** and **novelty**, so check those columns too. The two-stage model in notebook 06 combines both.

## 6. What did the embeddings learn?

Item vectors that point the same way are "similar" in taste space. If categories are available, project the item factors to 2-D with PCA and colour the largest categories. Clusters mean the model has rediscovered structure it never saw.

In [ ]:
from sklearn.decomposition import PCA

Y = als.item_factors_ / (np.linalg.norm(als.item_factors_, axis=1, keepdims=True) + 1e-9)
xy = PCA(n_components=2, random_state=0).fit_transform(Y)
fig, ax = plt.subplots(figsize=(6, 5))
if item_categories is not None:
    cats = pd.Series(data.item_map.to_raw(np.arange(len(Y)))).map(dict(zip(item_categories["itemid"], item_categories["categoryid"])))
    top_cats = cats.value_counts().index[:6]
    ax.scatter(xy[:, 0], xy[:, 1], s=4, color="#d6d5cf", label="other")
    for color, c in zip(PALETTE, top_cats):
        mask = (cats == c).to_numpy()
        ax.scatter(xy[mask, 0], xy[mask, 1], s=6, color=color, label=f"category {c}")
    ax.legend(markerscale=3, fontsize=8, loc="best")
else:
    ax.scatter(xy[:, 0], xy[:, 1], s=4, color=PALETTE[0])
ax.set_title("ALS item embeddings (PCA of normalised factors)")
ax.set_xticks([]); ax.set_yticks([])
plt.show()

In [ ]:
item = int(np.argmax(np.asarray((data.X_train > 0).sum(axis=0)).ravel()))
nbrs, sims = als.similar_items(item, n=5)
cat_of = dict(zip(item_categories["itemid"], item_categories["categoryid"])) if item_categories is not None else {}
raw = data.item_map.to_raw(item)
print(f"nearest neighbours of the most popular item {raw} (category {cat_of.get(raw, '?')}):")
for r, s in zip(data.item_map.to_raw(nbrs), sims):
    print(f"   {r:>8}  cos={s:.3f}  category={cat_of.get(r, '?')}")

## Key takeaways

* MF learns dense user and item embeddings, so it generalises beyond observed co-occurrences.
* **Implicit ALS** weights every cell by confidence and solves exact least squares, one row at a time.
* **BPR** optimises pairwise order, which is closer to what a ranked list needs.
* The same embeddings power "similar items", clustering and fast retrieval (approximate nearest neighbours) in production.

## Exercises

1. **(Easy)** Plot validation NDCG@10 against the number of ALS iterations (1–20). When does it stop improving?
2. **(Medium)** Replace `np.linalg.solve` in `_als_half_step` with a few steps of **conjugate gradient** (Takács et al., 2011). Measure the speed-up and the accuracy cost.
3. **(Medium)** BPR samples negatives uniformly, so most of them are "easy" (obscure items). Sample negatives in proportion to popularity instead. What happens to coverage and novelty?
4. **(Stretch)** Implement **WARP** loss: keep sampling negatives until one violates the margin, and scale the update by the estimated rank. Compare it with BPR on NDCG@10.